# ALS: фильтр 5/5 против 5/20

Сравниваем минимальное число пользователей у тайтла **5 и 20**;
минимум тайтлов у пользователя остаётся **5**. Итеративный фильтр применяется
только к train, до вычисления весов, как в прежних опытах.

Фиксируем score_coef=0, planned=0,5, остальные DEFAULT_STATUS_WEIGHTS,
оценки <5 исключаем, затухание выключено. ALS и seed прежние:
factors=64, regularization=0.1, alpha=20, iterations=20, num_threads=4;
seed 42/7/2026; confidence=1+alpha*weight.

Полная train-history используется для исключений. Сохраняем всех 7 184
warm-validation пользователей и все релевантные цели, даже если после фильтра
они недоступны модели. Fallback не добавляем: неизвестный пользователь получает
пустую выдачу и нулевое качество. Считаем доступность целей отдельно для диагностики.
Контрольная популярность за 28 дней обучается на прежнем train 5/5.
Test не оцениваем; последний период уже изучался в EDA, его независимость ограничена.
Исходные артефакты и defaults не меняем.

In [1]:
import os
os.environ["OPENBLAS_NUM_THREADS"] = "1"
import sys
import json
from pathlib import Path
from time import perf_counter
import importlib.metadata
import numpy as np
import pandas as pd
from IPython.display import display

sys.path.insert(0, str(Path.cwd().parent))
from app.training.datasets import load_anilist, filter_interactions
from app.training.evaluate import chronological_split, relevant_items, evaluate, ndcg_at_k
from app.training.weights import weighted_interactions, DEFAULT_STATUS_WEIGHTS
from app.recsys.popularity import PopularityRecommender
from app.recsys.als import ALSRecommender

pd.set_option("display.float_format", "{:,.6f}".format)
VALIDATION_START = pd.Timestamp("2026-09-02T00:34:23Z")
TEST_START = pd.Timestamp("2026-09-30T00:34:23Z")
SNAPSHOT_END = pd.Timestamp("2026-10-28T00:34:23Z")
MIN_ITEM_USERS = [5, 20]
SCORE_COEF = 0.0
BASE_STATUS_WEIGHTS = DEFAULT_STATUS_WEIGHTS.copy()
assert BASE_STATUS_WEIGHTS == dict(completed=1.0, watching=1.0, on_hold=0.5, planned=0.5, dropped=0.0)
SEEDS = [42, 7, 2026]
ALS_PARAMS = dict(factors=64, regularization=0.1, alpha=20, iterations=20, num_threads=4)
versions = {name: importlib.metadata.version(name) for name in ["implicit", "numpy", "scipy", "pandas"]}
print(versions)
inter = load_anilist()
assert inter.updated_at.max() == SNAPSHOT_END, "Снимок изменился: пересмотрите протокол"
history, validation, _ = chronological_split(
    inter, validation_start=VALIDATION_START, test_start=TEST_START)
training = filter_interactions(history, min_user_items=5, min_item_users=5)
counts = history.user_id.value_counts()
targets = {u: items for u, items in relevant_items(validation).items() if counts.get(u, 0) >= 5}
users = sorted(targets)
seen = history.groupby("user_id").anime_id.agg(set).to_dict()
print(f"Train после фильтра: {len(training):,} записей; warm-validation: {len(users):,} пользователей")
assert len(users) == 7184
assert len(history) == 3513393 and len(training) == 3499908
assert len(validation) == 218925

/home/deonik3/AniFolder/ml/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'implicit': '0.7.3', 'numpy': '2.5.4', 'scipy': '1.18.1', 'pandas': '3.1.0'}


Train после фильтра: 3,499,908 записей; warm-validation: 7,184 пользователей


## Контрольный baseline

Популярность за 28 дней на train 5/5: ожидаем NDCG@10 0,113021.

In [2]:
popular = PopularityRecommender(window_days=28).fit(training, now=VALIDATION_START, history=history)
popular_metrics = evaluate(popular.recommend_many(users, k=10), targets)
assert np.isclose(popular_metrics["ndcg@10"], 0.113021, atol=5e-7, rtol=0)
pd.Series(popular_metrics, name="популярность 28 дней").to_frame()

,популярность 28 дней
precision@10,0.099624
recall@10,0.064522
ndcg@10,0.113021
coverage@10,65.000000
users,"7,184.000000"


## Шесть запусков и доступность целей

Набор положительных пар может измениться — это эффект фильтра. Цели и история исключений остаются прежними.

In [3]:
records = []
filter_records = []
per_user = {}
for threshold in MIN_ITEM_USERS:
    filtered = filter_interactions(history, min_user_items=5, min_item_users=threshold)
    assert filtered.user_id.value_counts().min() >= 5
    assert filtered.anime_id.value_counts().min() >= threshold
    weighted = weighted_interactions(filtered, status_weights=BASE_STATUS_WEIGHTS.copy(),
                                     score_coef=SCORE_COEF, min_score=5, half_life_days=None)
    assert DEFAULT_STATUS_WEIGHTS == BASE_STATUS_WEIGHTS
    assert not weighted.status.eq("dropped").any()
    assert not weighted.score.lt(5).fillna(False).any()
    known_users = set(weighted.user_id)
    known_items = set(weighted.anime_id)
    target_pairs = sum(len(items) for items in targets.values())
    available_pairs = sum(len(items & known_items) if u in known_users else 0
                          for u, items in targets.items())
    filter_records.append({"min_item_users": threshold, "filtered_rows": len(filtered),
                           "filtered_users": filtered.user_id.nunique(),
                           "filtered_items": filtered.anime_id.nunique(),
                           "positive_rows": len(weighted), "model_users": len(known_users),
                           "model_items": len(known_items),
                           "unknown_validation_users": sum(u not in known_users for u in users),
                           "target_pairs": target_pairs, "available_target_pairs": available_pairs,
                           "available_target_fraction": available_pairs / target_pairs})
    for seed in SEEDS:
        started = perf_counter()
        model = ALSRecommender(**ALS_PARAMS, random_state=seed).fit(weighted, history=history)
        fit_seconds = perf_counter() - started
        started = perf_counter()
        recs = model.recommend_many(users, k=10)
        recommend_seconds = perf_counter() - started
        assert set(recs) == set(users)
        assert all(len(r) == len(set(r)) and not (set(r) & seen.get(u, set())) for u, r in recs.items())
        metrics = evaluate(recs, targets)
        per_user[(threshold, seed)] = np.array([ndcg_at_k(recs[u], targets[u], 10) for u in users])
        records.append({"min_item_users": threshold, "seed": seed, **metrics,
                        "training_rows": len(weighted),
                        "short_lists": sum(len(r) < 10 for r in recs.values()),
                        "fit_seconds": fit_seconds, "recommend_seconds": recommend_seconds})
        print(f"filter=5/{threshold}, seed={seed}: NDCG@10={metrics['ndcg@10']:.6f}, "
              f"coverage={metrics['coverage@10']}, train={fit_seconds:.1f}s", flush=True)
results = pd.DataFrame(records)
filter_stats = pd.DataFrame(filter_records).set_index("min_item_users")
display(filter_stats)
display(results.set_index(["min_item_users", "seed"]))

filter=5/5, seed=42: NDCG@10=0.100216, coverage=4253, train=7.4s


filter=5/5, seed=7: NDCG@10=0.099340, coverage=4219, train=6.8s


filter=5/5, seed=2026: NDCG@10=0.099730, coverage=4256, train=7.1s


filter=5/20, seed=42: NDCG@10=0.098854, coverage=4178, train=6.8s


filter=5/20, seed=7: NDCG@10=0.097956, coverage=4148, train=6.7s


filter=5/20, seed=2026: NDCG@10=0.099098, coverage=4175, train=6.7s


,filtered_rows,filtered_users,filtered_items,positive_rows,model_users,model_items,unknown_validation_users,target_pairs,available_target_pairs,available_target_fraction
min_item_users,,,,,,,,,,
5,3499908,8109,15799,3293589,8109,15799,0,155847,149461,0.959024
20,3440937,8106,10111,3239564,8106,10111,3,155847,147706,0.947763


precision@10  recall@10  ndcg@10  coverage@10  users  \
min_item_users seed                                                         
5              42        0.081974   0.064405 0.100216         4253   7184   
               7         0.081584   0.063838 0.099340         4219   7184   
               2026      0.081807   0.064172 0.099730         4256   7184   
20             42        0.080944   0.062804 0.098854         4178   7184   
               7         0.080331   0.062836 0.097956         4148   7184   
               2026      0.081403   0.063310 0.099098         4175   7184   

                     training_rows  short_lists  fit_seconds  \
min_item_users seed                                            
5              42          3293589            0     7.364696   
               7           3293589            0     6.838910   
               2026        3293589            0     7.112023   
20             42          3239564            3     6.757386   
               7           3239564            3     6.650174   
               2026        3239564            3     6.681845   

                     recommend_seconds  
min_item_users seed                     
5              42             9.119613  
               7              8.703379  
               2026           8.783299  
20             42             5.452882  
               7              5.522409  
               2026           5.948238

## Качество и устойчивость

Std — разброс трёх инициализаций на одном validation, не оценка статистической значимости.

In [4]:
summary = results.groupby("min_item_users").agg(
    ndcg_mean=("ndcg@10", "mean"), ndcg_std=("ndcg@10", "std"),
    ndcg_min=("ndcg@10", "min"), ndcg_max=("ndcg@10", "max"),
    precision_mean=("precision@10", "mean"), recall_mean=("recall@10", "mean"),
    coverage_mean=("coverage@10", "mean"))
display(summary)
print("Лучший порог по среднему NDCG:", summary.ndcg_mean.idxmax())
print("NDCG baseline популярности:", popular_metrics["ndcg@10"])
paired = results.pivot(index="seed", columns="min_item_users", values="ndcg@10")
paired_delta = (paired[20] - paired[5]).rename("ndcg_delta_20_minus_5")
display(paired_delta.to_frame())
delta = np.mean([per_user[(20, seed)] - per_user[(5, seed)] for seed in SEEDS], axis=0)
user_deltas = pd.Series({"mean_delta": delta.mean(), "лучше": int((delta > 1e-12).sum()),
                         "без изменения": int((np.abs(delta) <= 1e-12).sum()),
                         "хуже": int((delta < -1e-12).sum())})
display(user_deltas.to_frame("5/20 против 5/5"))

,ndcg_mean,ndcg_std,ndcg_min,ndcg_max,precision_mean,recall_mean,coverage_mean
min_item_users,,,,,,,
5,0.099762,0.000438,0.099340,0.100216,0.081788,0.064138,"4,242.666667"
20,0.098636,0.000601,0.097956,0.099098,0.080893,0.062983,"4,167.000000"


Лучший порог по среднему NDCG: 5
NDCG baseline популярности: 0.11302101779723082


,ndcg_delta_20_minus_5
seed,
7,-0.001384
42,-0.001362
2026,-0.000632


,5/20 против 5/5
mean_delta,-0.001126
лучше,"1,665.000000"
без изменения,"3,813.000000"
хуже,"1,706.000000"


## Артефакты опыта

Отдельная папка `ml/artifacts/als_filter_threshold/` (вне Git); таблицы также сохранены в выводах ноутбука.

In [5]:
out = Path.cwd().parent / "artifacts" / "als_filter_threshold"
out.mkdir(parents=True, exist_ok=True)
results.to_csv(out / "runs.csv", index=False)
summary.to_csv(out / "summary.csv")
filter_stats.to_csv(out / "filter_stats.csv")
paired_delta.to_csv(out / "paired_deltas.csv")
user_deltas.to_csv(out / "user_deltas.csv")
(out / "protocol.json").write_text(json.dumps({
    "snapshot_end": SNAPSHOT_END.isoformat(), "validation_start": VALIDATION_START.isoformat(),
    "test_start": TEST_START.isoformat(), "test_evaluated": False,
    "min_user_items": 5, "min_item_users": MIN_ITEM_USERS,
    "score_coef": SCORE_COEF, "status_weights": BASE_STATUS_WEIGHTS,
    "min_score": 5, "min_user_scores": 5, "clip": [0.5, 1.5], "half_life_days": None,
    "seeds": SEEDS, "als_params": ALS_PARAMS, "confidence": "1 + alpha * weight",
    "filter_before_weighting": True, "warm_validation_users": len(users),
    "target_filtering": False, "exclusion_history": "full train",
    "user_id_namespace": "anilist", "item_id_namespace": "anilist",
    "versions": versions, "popular_baseline": popular_metrics, "popular_filter": [5, 5],
}, ensure_ascii=False, indent=2) + "\n")
print("Сохранены:", ", ".join(p.name for p in sorted(out.iterdir())))

Сохранены: filter_stats.csv, paired_deltas.csv, protocol.json, runs.csv, summary.csv, user_deltas.csv


## Вывод

| Фильтр | Средний NDCG@10 | Std между seed | Precision@10 | Recall@10 | Среднее coverage@10 |
|---|---:|---:|---:|---:|---:|
| 5/5 | 0,099762 | 0,000438 | 0,081788 | 0,064138 | 4242,7 |
| 5/20 | 0,098636 | 0,000601 | 0,080893 | 0,062983 | 4167,0 |

- **Оставляем фильтр 5/5** для следующего опыта. Повышение порога до 20 снижает
  NDCG при каждом seed. Средняя разница −0,001126 (~−1,13%); precision, recall
  и абсолютное покрытие также ниже. Это результат на одном validation,
  а не доказательство статистической значимости или переносимости.
- Вариант 5/5 точно воспроизводит все метрики planned=0,5 из ноутбука 05
  на каждом seed. Все шесть запусков оценены на тех же 7 184 пользователях
  и 155 847 релевантных парах пользователь–тайтл.
- Фильтр 5/20 уменьшает каталог модели с **15 799 до 10 111 тайтлов** (~−36%),
  положительные взаимодействия — с 3 293 589 до 3 239 564 (~−1,64%).
  Число пользователей модели уменьшается с 8 109 до 8 106.
- **Три warm-validation пользователя становятся неизвестны модели** при 5/20:
  их пустые выдачи учтены как нулевое качество. При 5/5 полный топ-10 получают
  все 7 184; при 5/20 — 7 181 пользователь. Fallback отсутствует в обоих вариантах.
- Доля доступных модели релевантных пар (известны и пользователь, и тайтл)
  падает с 95,90% до 94,78%. Цели не отфильтрованы: это диагностический показатель,
  а не другой знаменатель для метрик качества.
- Выдача всех рекомендаций при 5/20 занимает в среднем ~5,6 с против ~8,9 с
  при 5/5 в этом окружении. Каталог меньше и выдача быстрее, но основная метрика
  выбора — NDCG@10 — ухудшается. Эти замеры не заменяют нагрузочный тест сервиса.
- Контрольная популярность 28 дней остаётся выше: **NDCG@10 0,113021**.
  Defaults, код модели и исходные артефакты не менялись; test не оценивался.
- Следующий шаг — настройки ALS на train 5/5, при score_coef=0 и planned=0,5.
  Сохраняем прежний временной протокол, полную историю исключений и цели validation.
